In [ ]:
#Kod do wczytywania danych i preprocessingu.
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.special import expit
import json
import time
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.utils import check_random_state

import cedalion
import cedalion.nirs
import cedalion.data
import cedalion.sigproc.quality as quality
import cedalion.sigproc.motion_correct as motion_correct
from cedalion.nirs import channel_distances
from cedalion import units

DATA_DIR    = r"/Users/antek/Desktop/Praca Magisterska kody/Dane"
FILE_PREFIX = "SITTING"
OUTDIR      = Path(r"/Users/antek/Desktop/Praca Magisterska kody/LOSO")
OUTDIR.mkdir(parents=True, exist_ok=True)

SNR_THRESHOLD = 10
FMIN, FMAX, BUTTER_ORDER = 0.01, 0.2, 4
DPF_VALUE = 6.0
BEFORE_S, AFTER_S = 0.0, 19.0

N_PERMUTATIONS = 200
RANDOM_SEED = 0

def strip_pint(da):
    vals = da.values
    if hasattr(vals, "magnitude"):
        vals = vals.magnitude
    coord_attrs = {k: da[k].attrs.copy() for k in da.coords}
    result = xr.DataArray(vals, dims=da.dims, coords=da.coords, attrs=da.attrs)
    for k, attrs in coord_attrs.items():
        result[k].attrs = attrs
    return result


def od_to_hb(od, geo3d):
    #Funkcja przejścia z optycznej gęstości (OD) na stężenia HbO i HbR, korzystając z danych Prahla. 
    #Napisana ręcznie z powodu występującego błędu w cedalion.nirs.od_to_hb z końcówki 2025 roku.
    od = strip_pint(od)
    path = cedalion.data.get("prahl_absorption_spectrum.tsv")
    with path.open("r") as fin:
        coeffs = np.loadtxt(fin, comments="#")
    wls = od.wavelength.values.astype(float)
    spectra = [interp1d(coeffs[:, 0], np.log(10) * coeffs[:, i] / 10) for i in [1, 2]]
    E_vals = np.array([s(wl) for s in spectra for wl in wls]).reshape(2, len(wls))
    E = xr.DataArray(E_vals, dims=["chromo", "wavelength"],
                     coords={"chromo": ["HbO", "HbR"], "wavelength": od.wavelength})
    Einv = xr.DataArray(np.linalg.pinv(E.values), dims=["wavelength", "chromo"],
                        coords={"wavelength": od.wavelength, "chromo": ["HbO", "HbR"]})
    dists = strip_pint(channel_distances(od, geo3d))
    dpf = xr.DataArray(np.full(len(wls), DPF_VALUE), dims=["wavelength"],
                       coords={"wavelength": od.wavelength})
    od_norm = od / (dists * dpf)
    hb = xr.dot(Einv, od_norm, dim="wavelength")
    hb["time"].attrs = od["time"].attrs
    return hb


def preprocess_one(file_path):
    #Funkcja przeprowadzająca pełen proces preprocessingu
    rec = cedalion.io.read_snirf(str(file_path))[0]
    rec.stim.cd.rename_events({"1": "MC", "2": "REST"})
    stim = rec.stim.sort_values("onset").reset_index(drop=True)
    stim = stim[stim["trial_type"].isin(["MC", "REST"])].reset_index(drop=True)
    onsets = stim["onset"].astype(float).to_numpy()
    durs = np.diff(onsets, append=onsets[-1])
    durs[-1] = max(1.0, durs[-2])
    stim["duration"] = durs

    amp = rec["amp"]
    amp = amp - float(amp.min()) + 1e-12
    _, snr_mask = quality.snr(amp, SNR_THRESHOLD)
    amp, _ = quality.prune_ch(amp, snr_mask.expand_dims(label=["snr"]), "any")

    od = cedalion.nirs.int2od(amp)
    od = motion_correct.tddr(od)
    od = motion_correct.motion_correct_wavelet(od)
    od = od.cd.freq_filter(fmin=FMIN, fmax=FMAX, butter_order=BUTTER_ORDER)

    hb = od_to_hb(od, rec.geo3d)
    channels = list(hb.channel.values.astype(str))
    return {"subject": file_path.stem, "hb": hb, "stim": stim, "channels": channels}


def extract_epochs_array(item, common_channels):
    hb = item["hb"].sel(channel=common_channels)
    epochs = hb.cd.to_epochs(
        item["stim"], ["MC", "REST"],
        before=BEFORE_S * units.s, after=AFTER_S * units.s,
    )
    epochs = epochs.transpose("epoch", "chromo", "channel", "reltime")
    X = epochs.values
    if hasattr(X, "magnitude"):
        X = X.magnitude
    y = epochs.trial_type.values.astype(str)
    finite_mask = np.all(np.isfinite(X.reshape(X.shape[0], -1)), axis=1)
    X, y = X[finite_mask], y[finite_mask]
    t = epochs.reltime.values.astype(float)
    fs = 1.0 / np.median(np.diff(t))
    subj_id = np.array([item["subject"]] * len(X))
    return X, y, subj_id, fs


#Ładowanie wszystkich plików
files = sorted(Path(DATA_DIR).glob(f"{FILE_PREFIX}*.snirf"))

cache = []
for f in files:
    try:
        cache.append(preprocess_one(f))

# wspólne kanały (intersection)
sets = [set(item["channels"]) for item in cache]
common_channels = sorted(set.intersection(*sets))


Xs, ys, ss, fs_list = [], [], [], []
for item in cache:
    try:
        X_, y_, s_, fs_ = extract_epochs_array(item, common_channels)
        if len(X_) > 0:
            Xs.append(X_); ys.append(y_); ss.append(s_); fs_list.append(fs_)

X_all   = np.concatenate(Xs, axis=0)     # (N, 2, K, T) chromo=[HbO,HbR]
y_all   = np.concatenate(ys, axis=0)     # "MC" / "REST"
subj_all = np.concatenate(ss, axis=0)
FS      = float(np.median(fs_list))
N_CHROMO, N_CHANNELS, N_SAMPLES = X_all.shape[1], X_all.shape[2], X_all.shape[3]


# Funkcja podsumowująca metryki klasyfikacji
def summary_metrics(y_true, y_pred, y_proba=None):
    mask = np.isin(y_true, ["MC", "REST"])
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"accuracy": np.nan, "balanced_accuracy": np.nan,
                "f1_macro": np.nan, "f1_MC": np.nan, "n": 0, "roc_auc": np.nan}
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro",
                             labels=["MC", "REST"], zero_division=0),
        "f1_MC": f1_score(y_true, y_pred, pos_label="MC", average="binary",
                          labels=["MC", "REST"], zero_division=0),
        "n": len(y_true), "roc_auc": np.nan,
    }
    if y_proba is not None and len(np.unique(y_true)) == 2:
        try:
            out["roc_auc"] = roc_auc_score((y_true == "MC").astype(int), y_proba[mask])
        except Exception:
            pass
    return out

#Funkcja przeprowadzająca ewaluację LOSO (Leave-One-Subject-Out) dla danego modelu, zwracająca metryki per-subject oraz ogólną dokładność.
def run_loso(X, y, subj, fit_predict_fn, method_name, need_proba=True):
    per, y_true_all, y_pred_all, y_proba_all = [], [], [], []
    for s in np.unique(subj):
        te, tr = (subj == s), (subj != s)
        try:
            out = fit_predict_fn(X[tr], y[tr], X[te])
        except Exception as e:
            print(f"  [LOSO] {method_name} | subject {s} FAILED: {e}")
            continue
        y_pred = out["y_pred"]
        y_proba = out.get("y_proba", None)
        m = summary_metrics(y[te], y_pred, y_proba if need_proba else None)
        m["subject"] = str(s)
        per.append(m)
        y_true_all.append(y[te]); y_pred_all.append(y_pred)
        if need_proba and y_proba is not None:
            y_proba_all.append(y_proba)

    per_df = pd.DataFrame(per)
    y_true_all = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_pred_all = np.concatenate(y_pred_all) if y_pred_all else np.array([])
    y_proba_all = (np.concatenate(y_proba_all)
                   if (need_proba and y_proba_all) else None)
    pooled_acc = accuracy_score(y_true_all, y_pred_all) if len(y_true_all) else np.nan
    return {
        "method": method_name,
        "per_subject": per_df,
        "y_true": y_true_all, "y_pred": y_pred_all, "y_proba": y_proba_all,
        "pooled_acc": pooled_acc,
    }


In [ ]:
#Funkcja przeprowadzająca pełen eksperyment porównujący wyniki klasyfikacji LOSO dla różnych modeli (OLS, OLS z SS, AR-IRLS) i różnych funkcji HRF, zapisująca wyniki do pliku CSV.
#Zwraca podsumowanie wyników w formie DataFrame oraz zapisuje szczegółowe wyniki per-subject do pliku CSV.
#Funkcjonalność dla kanałów krótkich (SS) w tym kodzie nie działa prawidłowo i jest naprawiana w następnym bloku, gdzie stworzone tutaj .csv są nadpisywane.

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from tqdm import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              precision_score, recall_score, f1_score)

import cedalion.io
import cedalion.nirs
import cedalion.sigproc.quality as quality
import cedalion.sigproc.motion_correct as motion_correct
import cedalion.models.glm as glm
from cedalion.models.glm import basis_functions as bf
from cedalion.nirs import channel_distances


#Dla metody AR-IRLS nie stosujemy filtru bandpass, który jest standardem dla OLS, ale może zakłócać modelowanie szumu w AR-IRLS.
USE_BANDPASS = False

DRIFT_LIST = [0, 1]

HRF_LIST = [
    ("Gamma_t0_s5",    lambda: bf.Gamma(tau=0*units.s, sigma=5*units.s)),
    ("Gamma_t0_s3",    lambda: bf.Gamma(tau=0*units.s, sigma=3*units.s)),
    ("Gamma_t2_s4",    lambda: bf.Gamma(tau=2*units.s, sigma=4*units.s)),
    ("GammaDeriv",     lambda: bf.GammaDeriv(tau=0*units.s, sigma=5*units.s)),
    ("AFNIGamma",      lambda: bf.AFNIGamma(p=8.6, q=0.547*units.s)),
    ("GaussKernels",   lambda: bf.GaussianKernels(
                            t_pre=0*units.s, t_post=19*units.s,
                            t_delta=3*units.s, t_std=2*units.s)),
]

NOISE_MODELS = [
    ("OLS",    "ols",     False),  
    ("OLS_SS", "ols",     True),
    ("AR_IRLS","ar_irls", False),
]

MAX_DUR_S  = 19.0
OUTDIR_RES = Path(r"/Users/antek/Desktop/Praca Magisterska kody/LOSO Results")
OUTDIR_BET = OUTDIR_RES / "betas"
OUTDIR_RES.mkdir(parents=True, exist_ok=True)
OUTDIR_BET.mkdir(parents=True, exist_ok=True)


def preprocess_subject(file_path, use_bandpass):
    rec = cedalion.io.read_snirf(str(file_path))[0]
    rec.stim.cd.rename_events({"1": "MC", "2": "REST"})
    stim = rec.stim.sort_values("onset").reset_index(drop=True)
    stim = stim[stim["trial_type"].isin(["MC", "REST"])].reset_index(drop=True)
    onsets = stim["onset"].astype(float).to_numpy()
    durs   = np.diff(onsets, append=onsets[-1])
    durs[-1] = max(1.0, durs[-2])
    stim["duration"] = durs

    amp = rec["amp"]
    amp = amp - float(amp.min()) + 1e-12
    _, snr_mask = quality.snr(amp, SNR_THRESHOLD)
    amp, _ = quality.prune_ch(amp, snr_mask.expand_dims(label=["snr"]), "any")

    od = cedalion.nirs.int2od(amp)
    od = motion_correct.tddr(od)
    od = motion_correct.motion_correct_wavelet(od)

    if use_bandpass:
        od = od.cd.freq_filter(fmin=0.01, fmax=0.2, butter_order=4)

    hb = od_to_hb(od, rec.geo3d)   # z Cell 1

    dists    = strip_pint(channel_distances(hb, rec.geo3d))
    short_ch = list(dists.channel.values[dists.values < 0.015].astype(str))
    all_ch   = list(hb.channel.values.astype(str))
    long_ch  = [c for c in all_ch if c not in short_ch]

    return {
        "subject":        file_path.stem,
        "hb":             hb,
        "stim":           stim,
        "long_channels":  long_ch,
        "short_channels": short_ch,
        "geo3d":          rec.geo3d,
    }



_files = sorted(Path(DATA_DIR).glob(f"{FILE_PREFIX}*.snirf"))

cache_v2 = []
for _f in _files:
    try:
        cache_v2.append(preprocess_subject(_f, USE_BANDPASS))

common_long_v2 = sorted(set.intersection(*[
    set(item["long_channels"]) for item in cache_v2
]))


def _prep_ep(ep):
    if "reltime" in ep.dims:
        ep = ep.rename({"reltime": "time"})
    if "samples" not in ep.coords:
        ep = ep.assign_coords(samples=("time", np.arange(ep.sizes["time"])))
    if "units" not in ep["time"].attrs:
        ep["time"].attrs["units"] = "s"
    return ep


def extract_beta_epoch(ep_long, ep_short, dur_s, hrf_basis,
                        drift_order, geo3d, use_ss, noise_model):
    ep_l = _prep_ep(ep_long)
    ep_s = _prep_ep(ep_short) if ep_short is not None else None

    stim_one = pd.DataFrame({
        "onset": [0.0], "duration": [dur_s],
        "trial_type": ["X"], "value": [1.0],
    })
    dms = (
        glm.design_matrix.hrf_regressors(ep_l, stim_one, hrf_basis)
        & glm.design_matrix.drift_regressors(ep_l, drift_order=drift_order)
    )
    if use_ss and ep_s is not None and ep_s.sizes.get("channel", 0) > 0:
        dms = dms & glm.design_matrix.closest_short_channel_regressor(
            ts_long=ep_l, ts_short=ep_s, geo3d=geo3d,
        )
    betas     = glm.fit(ep_l, dms, noise_model=noise_model).sm.params
    reg_names = betas.regressor.values.astype(str)
    hrf_mask  = np.array(["HRF" in r for r in reg_names])
    return betas.isel(regressor=hrf_mask).values.reshape(-1)


def beta_csv_path(nm_name, hrf_name, drift):
    return OUTDIR_BET / f"{nm_name}__{hrf_name}__d{drift}.csv"

_avg_ep = int(np.mean([
    len(item["stim"][item["stim"]["trial_type"].isin(["MC", "REST"])])
    for item in cache_v2
]))
_combos_todo = []
for nm_name, nm_code, use_ss in NOISE_MODELS:
    for hrf_name, _ in HRF_LIST:
        for drift in DRIFT_LIST:
            if not beta_csv_path(nm_name, hrf_name, drift).exists():
                _combos_todo.append((nm_name, nm_code, use_ss, hrf_name, drift))

_total_ep = len(_combos_todo) * len(cache_v2) * _avg_ep
_total_combos = len(NOISE_MODELS) * len(HRF_LIST) * len(DRIFT_LIST)


run_i = 0
with tqdm(total=_total_ep, unit="ep", ncols=100,
          bar_format="{desc} |{bar}| {n_fmt}/{total_fmt} ep "
                     "[{elapsed}<{remaining}  {rate_fmt}]") as pbar:

    for nm_name, nm_code, use_ss in NOISE_MODELS:
        for hrf_name, hrf_fn in HRF_LIST:
            for drift in DRIFT_LIST:

                out_path = beta_csv_path(nm_name, hrf_name, drift)
                run_i += 1
                combo = f"{nm_name}__{hrf_name}__d{drift}"

                if out_path.exists():
                    pbar.write(f"  [skip – plik istnieje] {combo}")
                    pbar.update(len(cache_v2) * _avg_ep)
                    continue

                pbar.set_description(
                    f"[{run_i:02d}/{_total_combos}] {combo:<42}"
                )

                rows = []
                t0   = time.time()

                for item in cache_v2:
                    subj = item["subject"]
                    try:
                        hb_ls = item["hb"].sel(channel=common_long_v2)
                    except Exception as _e:
                        pbar.write(f"  [skip ch] {subj}: {_e}"); continue

                    hb_ss = None
                    if use_ss and item["short_channels"]:
                        try:
                            hb_ss = item["hb"].sel(channel=item["short_channels"])
                        except Exception:
                            pass

                    try:
                        ep_ls = hb_ls.cd.to_epochs(
                            item["stim"], ["MC", "REST"],
                            before=BEFORE_S * units.s, after=AFTER_S * units.s,
                        )
                        ep_ss = hb_ss.cd.to_epochs(
                            item["stim"], ["MC", "REST"],
                            before=BEFORE_S * units.s, after=AFTER_S * units.s,
                        ) if hb_ss is not None else None
                    except Exception as _e:
                        pbar.write(f"  [skip epoch] {subj}: {_e}"); continue

                    for i in range(ep_ls.sizes["epoch"]):
                        lab = str(ep_ls.trial_type.values[i])
                        if lab not in ("MC", "REST"):
                            pbar.update(1); continue
                        try:
                            dur  = float(np.clip(
                                float(item["stim"].iloc[i]["duration"])
                                if i < len(item["stim"]) else MAX_DUR_S,
                                1.0, MAX_DUR_S,
                            ))
                            beta = extract_beta_epoch(
                                ep_ls.isel(epoch=i),
                                ep_ss.isel(epoch=i) if ep_ss is not None else None,
                                dur_s=dur, hrf_basis=hrf_fn(),
                                drift_order=drift, geo3d=item["geo3d"],
                                use_ss=use_ss, noise_model=nm_code,
                            )
                        except Exception as _e:
                            pbar.write(f"  [skip ep] {subj} ep={i}: {_e}")
                            pbar.update(1); continue

                        if beta is None or not np.all(np.isfinite(beta)):
                            pbar.update(1); continue

                        row = {"subject": subj, "label": lab,
                               "hrf": hrf_name, "drift": drift,
                               "noise_model": nm_name}
                        for bi, bv in enumerate(beta):
                            row[f"b{bi:04d}"] = bv
                        rows.append(row)
                        pbar.update(1)

                if rows:
                    pd.DataFrame(rows).to_csv(out_path, index=False)
                    elapsed = time.time() - t0
                    pbar.write(
                        f"  OK  {combo:<42}  "
                        f"n_ep={len(rows)}  t={elapsed/60:.1f}min  → {out_path.name}"
                    )
                else:
                    pbar.write(f"  BRAK DANYCH: {combo}")




def run_loso_from_df(df):
    beta_cols = [c for c in df.columns if c.startswith("b") and c[1:].isdigit()]
    rows, y_all, yp_all = [], [], []

    for s in df["subject"].unique():
        te_mask = df["subject"] == s
        tr_mask = ~te_mask
        X_tr = df.loc[tr_mask, beta_cols].values.astype(float)
        y_tr = df.loc[tr_mask, "label"].values
        X_te = df.loc[te_mask, beta_cols].values.astype(float)
        y_te = df.loc[te_mask, "label"].values

        # usuń wiersze z NaN (różne długości beta dla różnych HRF)
        ok_tr = np.all(np.isfinite(X_tr), axis=1)
        ok_te = np.all(np.isfinite(X_te), axis=1)
        X_tr, y_tr = X_tr[ok_tr], y_tr[ok_tr]
        X_te, y_te = X_te[ok_te], y_te[ok_te]

        if len(np.unique(y_tr)) < 2 or len(y_te) == 0:
            continue

        pipe = Pipeline([
            ("sc",  StandardScaler()),
            ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
        ])
        try:
            pipe.fit(X_tr, y_tr)
            y_pred = pipe.predict(X_te)

        y_all.append(y_te); yp_all.append(y_pred)
        rows.append({
            "subject":           str(s),
            "accuracy":          accuracy_score(y_te, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_te, y_pred),
            "precision":         precision_score(y_te, y_pred, pos_label="MC", zero_division=0),
            "recall":            recall_score(y_te, y_pred, pos_label="MC", zero_division=0),
            "f1_MC":             f1_score(y_te, y_pred, pos_label="MC", average="binary", zero_division=0),
        })

    pooled = (
        accuracy_score(np.concatenate(y_all), np.concatenate(yp_all))
        if y_all else np.nan
    )
    return pd.DataFrame(rows), pooled


NM_LABEL = {
    "OLS":    "GLM OLS",
    "OLS_SS": "GLM OLS SS",
    "AR_IRLS":"GLM AR IRLS",
}

all_loso_results = []

for nm_name, _, _ in NOISE_MODELS:
    loso_rows = []

    for hrf_name, _ in HRF_LIST:
        for drift in DRIFT_LIST:
            fpath = beta_csv_path(nm_name, hrf_name, drift)
            if not fpath.exists():
                print(f"    [brak pliku] {fpath.name}"); continue

            df_beta = pd.read_csv(fpath)
            df_subj, pooled = run_loso_from_df(df_beta)

            for _, row in df_subj.iterrows():
                loso_rows.append({
                    "noise_model": nm_name,
                    "hrf":         hrf_name,
                    "drift":       drift,
                    "combo":       f"{hrf_name}_d{drift}",
                    **row.to_dict(),
                })
            all_loso_results.append({
                "noise_model": nm_name, "hrf": hrf_name, "drift": drift,
                "combo":       f"{hrf_name}_d{drift}",
                "pooled_acc":  pooled,
                "mean_acc":    df_subj["accuracy"].mean(),
                "std_acc":     df_subj["accuracy"].std(),
                "mean_f1":     df_subj["f1_MC"].mean(),
            })


    if loso_rows:
        out_csv = OUTDIR_RES / f"{NM_LABEL[nm_name]}.csv"
        pd.DataFrame(loso_rows).to_csv(out_csv, index=False)

df_summary = pd.DataFrame(all_loso_results)
df_summary.to_csv(OUTDIR_RES / "summary_all.csv", index=False)


_colors = {"OLS": "darkorange", "OLS_SS": "steelblue", "AR_IRLS": "seagreen"}
_combos_uniq = df_summary["combo"].unique()
_x = np.arange(len(_combos_uniq))
_w = 0.25

fig, ax = plt.subplots(figsize=(len(_combos_uniq) * 1.3 + 2, 5))
for k, (nm_name, _, _) in enumerate(NOISE_MODELS):
    _sub = df_summary[df_summary["noise_model"] == nm_name]
    _vals = [
        _sub[_sub["combo"] == c]["pooled_acc"].values
        for c in _combos_uniq
    ]
    _vals = [v[0] if len(v) > 0 else np.nan for v in _vals]
    ax.bar(_x + (k-1)*_w, _vals, _w,
           label=NM_LABEL[nm_name], color=_colors[nm_name], alpha=0.87)

ax.axhline(0.5, color="red", ls="--", lw=1.2, label="chance")
ax.set_xticks(_x)
ax.set_xticklabels(_combos_uniq, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Pooled LOSO accuracy")
ax.set_ylim(0, 1)
ax.set_title(
    "GLM OLS vs OLS+SS vs AR-IRLS  —  pooled LOSO accuracy\n"
    f"(preprocessing: {'z bandpass' if USE_BANDPASS else 'bez bandpass'})",
    fontweight="bold",
)
ax.legend()
plt.tight_layout()
fig.savefig(OUTDIR_RES / "porownanie_OLS_SS_ARIRLS.pdf", format="pdf")
plt.show(); plt.close(fig)



In [ ]:
#Komórka adresująca problem klasyfikacji metodą GLM OLS SS
#W tej komórce dokonujemy ponownej ekstrakcji beta dla metody OLS_SS, ale tym razem korzystając z poprawnego zestawu kanałów długich (>=15mm) wspólnych dla wszystkich uczestników.
#Następnie ponownie przeprowadzamy klasyfikację LOSO i zapisujemy wyniki do plików CSV, nadpisując poprzednie wyniki, które były oparte na błędnym zestawie kanałów.

import numpy as np
import pandas as pd
import time
import warnings
from pathlib import Path
from tqdm import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis)
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score)

import cedalion.models.glm as glm
from cedalion.models.glm import basis_functions as bf
from cedalion.nirs import channel_distances

warnings.filterwarnings("ignore")

OUTDIR_RES = Path(r"/Users/antek/Desktop/Praca Magisterska kody/LOSO Results")
OUTDIR_BET = OUTDIR_RES / "betas"
OUTDIR_CLF = OUTDIR_RES / "classifiers"
OUTDIR_CLF.mkdir(parents=True, exist_ok=True)


SHORT_THRESHOLD_MM = 15.0    #kluczowa zmiana wzgledem poprzedniego bloku kodu, błąd w zrozumieniu jednostek uzywanych przez cedalion
MAX_DUR_S          = 19.0
DRIFT_LIST         = [0, 1]
HRF_LIST = [
    ("Gamma_t0_s5",  lambda: bf.Gamma(tau=0*units.s, sigma=5*units.s)),
    ("Gamma_t0_s3",  lambda: bf.Gamma(tau=0*units.s, sigma=3*units.s)),
    ("Gamma_t2_s4",  lambda: bf.Gamma(tau=2*units.s, sigma=4*units.s)),
    ("GammaDeriv",   lambda: bf.GammaDeriv(tau=0*units.s, sigma=5*units.s)),
    ("AFNIGamma",    lambda: bf.AFNIGamma(p=8.6, q=0.547*units.s)),
    ("GaussKernels", lambda: bf.GaussianKernels(
                        t_pre=0*units.s, t_post=19*units.s,
                        t_delta=3*units.s, t_std=2*units.s)),
]


def split_channels(item):
    d_mm = channel_distances(item["hb"], item["geo3d"]).pint.to("mm").pint.magnitude
    ch   = item["hb"].channel.values.astype(str)
    short = list(ch[d_mm <  SHORT_THRESHOLD_MM])
    long_ = list(ch[d_mm >= SHORT_THRESHOLD_MM])
    return short, long_

_long_sets = []
for _it in cache_v2:
    _short, _long = split_channels(_it)
    _long_sets.append(set(_long))
common_long_ss = sorted(set.intersection(*_long_sets))


for _it in cache_v2:
    _s, _l = split_channels(_it)
    print(f"  {_it['subject']:<12} krótkich={len(_s):>2}  długich={len(_l):>2}")

def _prep_ep(ep):
    if "reltime" in ep.dims:
        ep = ep.rename({"reltime": "time"})
    if "samples" not in ep.coords:
        ep = ep.assign_coords(samples=("time", np.arange(ep.sizes["time"])))
    if "units" not in ep["time"].attrs:
        ep["time"].attrs["units"] = "s"
    return ep

def extract_beta_epoch_ss(ep_long, ep_short, dur_s, hrf_basis,
                          drift_order, geo3d):
    ep_l = _prep_ep(ep_long)
    ep_s = _prep_ep(ep_short) if ep_short is not None else None
    stim_one = pd.DataFrame({"onset": [0.0], "duration": [dur_s],
                             "trial_type": ["X"], "value": [1.0]})
    dms = (glm.design_matrix.hrf_regressors(ep_l, stim_one, hrf_basis)
           & glm.design_matrix.drift_regressors(ep_l, drift_order=drift_order))
    if ep_s is not None and ep_s.sizes.get("channel", 0) > 0:
        dms = dms & glm.design_matrix.closest_short_channel_regressor(
            ts_long=ep_l, ts_short=ep_s, geo3d=geo3d)
    betas    = glm.fit(ep_l, dms, noise_model="ols").sm.params
    reg_str  = betas.regressor.values.astype(str)
    hrf_mask = np.array(["HRF" in r for r in reg_str])
    return betas.isel(regressor=hrf_mask).values.reshape(-1)


_combos = [(h, hf, d) for (h, hf) in HRF_LIST for d in DRIFT_LIST]
for hrf_name, hrf_fn, drift in tqdm(_combos, desc="OLS_SS combos", unit="combo"):
    out_path = OUTDIR_BET / f"OLS_SS__{hrf_name}__d{drift}.csv"
    rows = []
    for item in cache_v2:
        subj          = item["subject"]
        short_ch, _   = split_channels(item)
        try:
            hb_ls = item["hb"].sel(channel=common_long_ss)
        hb_ss = item["hb"].sel(channel=short_ch) if short_ch else None
        try:
            ep_ls = hb_ls.cd.to_epochs(item["stim"], ["MC", "REST"],
                        before=BEFORE_S*units.s, after=AFTER_S*units.s)
            ep_ss = (hb_ss.cd.to_epochs(item["stim"], ["MC", "REST"],
                        before=BEFORE_S*units.s, after=AFTER_S*units.s)
                     if hb_ss is not None else None)


        for i in range(ep_ls.sizes["epoch"]):
            lab = str(ep_ls.trial_type.values[i])
            if lab not in ("MC", "REST"):
                continue
            try:
                dur  = float(np.clip(
                    float(item["stim"].iloc[i]["duration"])
                    if i < len(item["stim"]) else MAX_DUR_S, 1.0, MAX_DUR_S))
                beta = extract_beta_epoch_ss(
                    ep_ls.isel(epoch=i),
                    ep_ss.isel(epoch=i) if ep_ss is not None else None,
                    dur_s=dur, hrf_basis=hrf_fn(),
                    drift_order=drift, geo3d=item["geo3d"])
            except Exception as e:
                print(f"  [skip ep] {subj} ep={i}: {e}"); continue
            if beta is None or not np.all(np.isfinite(beta)):
                continue
            row = {"subject": subj, "label": lab, "hrf": hrf_name,
                   "drift": drift, "noise_model": "OLS_SS"}
            for bi, bv in enumerate(beta):
                row[f"b{bi:04d}"] = bv
            rows.append(row)

    if rows:
        pd.DataFrame(rows).to_csv(out_path, index=False)
        tqdm.write(f"  OK  OLS_SS__{hrf_name}__d{drift}  "
                   f"n_ep={len(rows)}  n_feat={len(rows[0])-5}  → {out_path.name}")
    else:
        tqdm.write(f"  BRAK DANYCH: OLS_SS__{hrf_name}__d{drift}")


def _pca(n): return ("pca", PCA(n_components=n, whiten=True))

CLASSIFIERS = [
    ("LDA (shrinkage)",  Pipeline([("sc", StandardScaler()),
        ("clf", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))])),
    ("LDA (svd)",        Pipeline([("sc", StandardScaler()),
        ("clf", LinearDiscriminantAnalysis(solver="svd"))])),
    ("QDA (+PCA30)",     Pipeline([("sc", StandardScaler()), _pca(30),
        ("clf", QuadraticDiscriminantAnalysis())])),
    ("LogReg L2",        Pipeline([("sc", StandardScaler()),
        ("clf", LogisticRegression(C=1.0, max_iter=2000, random_state=42))])),
    ("LogReg L1",        Pipeline([("sc", StandardScaler()),
        ("clf", LogisticRegression(C=0.1, penalty="l1", solver="liblinear",
                                   max_iter=2000, random_state=42))])),
    ("Ridge",            Pipeline([("sc", StandardScaler()),
        ("clf", RidgeClassifier(alpha=1.0))])),
    ("SVM rbf C=1",      Pipeline([("sc", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",
                    probability=True, random_state=42))])),
    ("SVM linear C=0.1", Pipeline([("sc", StandardScaler()),
        ("clf", SVC(kernel="linear", C=0.1, probability=True, random_state=42))])),
    ("LinearSVC C=0.1",  Pipeline([("sc", StandardScaler()),
        ("clf", LinearSVC(C=0.1, max_iter=5000, random_state=42))])),
    ("kNN k=11 (+PCA30)",Pipeline([("sc", StandardScaler()), _pca(30),
        ("clf", KNeighborsClassifier(n_neighbors=11))])),
    ("GaussianNB (+PCA30)", Pipeline([("sc", StandardScaler()), _pca(30),
        ("clf", GaussianNB())])),
    ("RandomForest 300", Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, max_features="sqrt",
                                       random_state=42, n_jobs=-1))])),
    ("ExtraTrees 200",   Pipeline([
        ("clf", ExtraTreesClassifier(n_estimators=200, max_features="sqrt",
                                     random_state=42, n_jobs=-1))])),
    ("GradBoosting 100 (+PCA30)", Pipeline([("sc", StandardScaler()), _pca(30),
        ("clf", GradientBoostingClassifier(n_estimators=100, random_state=42))])),
    ("MLP 100-50",       Pipeline([("sc", StandardScaler()),
        ("clf", MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500,
                              early_stopping=True, random_state=42))])),
    ("SGD hinge",        Pipeline([("sc", StandardScaler()),
        ("clf", SGDClassifier(loss="hinge", alpha=0.001,
                              max_iter=1000, random_state=42))])),
]

def run_loso(df, pipe):
    bc = [c for c in df.columns if c.startswith("b") and c[1:].isdigit()]
    rows, y_all, yp_all = [], [], []
    for s in df["subject"].unique():
        te = df["subject"] == s
        X_tr = df.loc[~te, bc].values.astype(float); y_tr = df.loc[~te, "label"].values
        X_te = df.loc[ te, bc].values.astype(float); y_te = df.loc[ te, "label"].values
        ok_tr = np.all(np.isfinite(X_tr), axis=1); ok_te = np.all(np.isfinite(X_te), axis=1)
        X_tr, y_tr = X_tr[ok_tr], y_tr[ok_tr]; X_te, y_te = X_te[ok_te], y_te[ok_te]
        if len(np.unique(y_tr)) < 2 or len(y_te) == 0:
            continue
        try:
            pipe.fit(X_tr, y_tr); y_pred = pipe.predict(X_te)
            try:
                sc = (pipe.predict_proba(X_te)[:, 1] if hasattr(pipe, "predict_proba")
                      else pipe.decision_function(X_te))
                auc = roc_auc_score((y_te == "MC").astype(int), sc)
            except Exception:
                auc = np.nan
        except Exception:
            continue
        y_all.append(y_te); yp_all.append(y_pred)
        rows.append({"subject": str(s),
                     "accuracy": accuracy_score(y_te, y_pred),
                     "balanced_accuracy": balanced_accuracy_score(y_te, y_pred),
                     "f1_MC": f1_score(y_te, y_pred, pos_label="MC", zero_division=0),
                     "auc": auc})
    pooled = (accuracy_score(np.concatenate(y_all), np.concatenate(yp_all))
              if y_all else np.nan)
    return pd.DataFrame(rows), pooled


beta_files = sorted(OUTDIR_BET.glob("*.csv"))

all_rows = []
for fpath in tqdm(beta_files, desc="Pliki beta", unit="plik"):
    nm, hrf, dstr = fpath.stem.split("__")
    drift = int(dstr.lstrip("d"))
    df_b  = pd.read_csv(fpath)
    for clf_name, pipe in CLASSIFIERS:
        df_s, pooled = run_loso(df_b, pipe)
        all_rows.append({
            "noise_model": nm, "hrf": hrf, "drift": drift, "combo": fpath.stem,
            "classifier": clf_name, "pooled_acc": pooled,
            "mean_acc":     df_s["accuracy"].mean()          if len(df_s) else np.nan,
            "std_acc":      df_s["accuracy"].std()           if len(df_s) else np.nan,
            "mean_bal_acc": df_s["balanced_accuracy"].mean() if len(df_s) else np.nan,
            "mean_f1":      df_s["f1_MC"].mean()             if len(df_s) else np.nan,
            "mean_auc":     df_s["auc"].mean()               if len(df_s) else np.nan,
        })

df_res = pd.DataFrame(all_rows)
df_res.to_csv(OUTDIR_CLF / "all_classifiers_results.csv", index=False)

# ── Podsumowania ──────────────────────────────────────────────────────────────
by_clf = (df_res.groupby("classifier")["pooled_acc"]
          .agg(["mean", "std", "max"]).sort_values("mean", ascending=False))

for nm in df_res["noise_model"].unique():
    sub = df_res[df_res["noise_model"] == nm]
    best = sub.loc[sub["pooled_acc"].idxmax()]





In [ ]:
#Budowanie klasyfikatorów dla jednego wybranego pipelinu funkcji HRF i klasyfikatora. 
#Przeprowadzono równiez porównanie metod za pomocą testu Friedmana
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score)

OUTDIR_RES = Path(r"/Users/antek/Desktop/Praca Magisterska kody/LOSO Results")
OUTDIR_BET = OUTDIR_RES / "betas"
OUTDIR_CMP = OUTDIR_RES / "porownanie_3metod_Gamma_t2_s4_d0"
OUTDIR_CMP.mkdir(parents=True, exist_ok=True)

HRF_NAME  = "Gamma_t2_s4"
DRIFT     = 0
POS_LABEL = "MC"

METHODS = [        
    ("OLS",     "GLM OLS"),
    ("OLS_SS",  "GLM OLS+SS"),
    ("AR_IRLS", "GLM AR-IRLS"),
]

def beta_path(nm):
    return OUTDIR_BET / f"{nm}__{HRF_NAME}__d{DRIFT}.csv"

def make_slda():
    return Pipeline([("sc", StandardScaler()),
                     ("clf", LinearDiscriminantAnalysis(solver="lsqr",
                                                        shrinkage="auto"))])

def run_loso(df, pipe):
    bc = [c for c in df.columns if c.startswith("b") and c[1:].isdigit()]
    rows = []
    for s in df["subject"].unique():
        te = df["subject"] == s
        X_tr = df.loc[~te, bc].values.astype(float); y_tr = df.loc[~te, "label"].values
        X_te = df.loc[ te, bc].values.astype(float); y_te = df.loc[ te, "label"].values
        ok_tr = np.all(np.isfinite(X_tr), axis=1); ok_te = np.all(np.isfinite(X_te), axis=1)
        X_tr, y_tr = X_tr[ok_tr], y_tr[ok_tr]; X_te, y_te = X_te[ok_te], y_te[ok_te]
        if len(np.unique(y_tr)) < 2 or len(y_te) == 0:
            continue
        try:
            pipe.fit(X_tr, y_tr); y_pred = pipe.predict(X_te)
            try:
                cls = list(pipe.classes_)
                if hasattr(pipe, "predict_proba"):
                    sc = pipe.predict_proba(X_te)[:, cls.index(POS_LABEL)]
                else:
                    sc = pipe.decision_function(X_te)
                auc = roc_auc_score((y_te == POS_LABEL).astype(int), sc)
            except Exception:
                auc = np.nan
        except Exception:
            continue
        rows.append({"subject": str(s),
                     "accuracy":          accuracy_score(y_te, y_pred),
                     "balanced_accuracy": balanced_accuracy_score(y_te, y_pred),
                     "f1_MC": f1_score(y_te, y_pred, pos_label=POS_LABEL, zero_division=0),
                     "auc": auc})
    return pd.DataFrame(rows)

per_subj = {}   
for nm_key, nm_lbl in METHODS:
    fp = beta_path(nm_key)
    if not fp.exists():
        print(f"  [BRAK PLIKU] {fp.name} — pomijam {nm_lbl}")
        continue
    df_b = pd.read_csv(fp)
    res  = run_loso(df_b, make_slda())
    per_subj[nm_lbl] = res


acc_wide = (pd.concat({k: v.set_index("subject")["accuracy"]
                       for k, v in per_subj.items()}, axis=1)
            .dropna())
method_names = list(acc_wide.columns)
subjects_raw = list(acc_wide.index)
N, k = acc_wide.shape

pacjent_map = {s: f"Pacjent {i+1}" for i, s in enumerate(subjects_raw)}
acc_disp = acc_wide.rename(index=pacjent_map)


acc_disp.to_csv(OUTDIR_CMP / "accuracy_per_pacjent.csv")


desc = pd.DataFrame({
    "mean":   acc_wide.mean(),
    "std":    acc_wide.std(),
    "median": acc_wide.median(),
    "IQR":    acc_wide.quantile(.75) - acc_wide.quantile(.25),
    "min":    acc_wide.min(),
    "max":    acc_wide.max(),
}).sort_values("mean", ascending=False)


best_method = desc.index[0]
desc.to_csv(OUTDIR_CMP / "statystyki_opisowe.csv")


chi2, p_fried = stats.friedmanchisquare(*[acc_wide[m].values for m in method_names])
df_fried = k - 1
kendall_w = chi2 / (N * (k - 1))      # effect size Kendalla W (0–1)
mean_ranks = acc_wide.rank(axis=1, ascending=False).mean().sort_values()

print("  TEST FRIEDMANA  (H0: wszystkie metody dają ten sam rozkład accuracy)")
print(f"  N (pacjenci)         = {N}")
print(f"  k (metody)           = {k}")
print(f"  chi^2 (Friedman)     = {chi2:.4f}")
print(f"  df                   = {df_fried}")
print(f"  p-value              = {p_fried:.4g}")
for m, r in mean_ranks.items():
    print(f"      {m:<14} {r:.2f}")
print(f"  Wniosek: {'ISTOTNA różnica (p<0.05)' if p_fried<0.05 else 'BRAK istotnej różnicy (p>=0.05)'}")


pairs = list(combinations(method_names, 2))
n_comp = len(pairs)
wil_rows = []
for a, b in pairs:
    da, db = acc_wide[a].values, acc_wide[b].values
    diff = da - db
    if np.allclose(diff, 0):
        W, p, z = np.nan, 1.0, 0.0
    else:
        res = stats.wilcoxon(da, db, zero_method="wilcox", method="approx")
        W, p = res.statistic, res.pvalue
        z = res.zstatistic
    r_eff = abs(z) / np.sqrt(N) if N > 0 else np.nan      # effect size r = |Z|/sqrt(N)
    wil_rows.append({
        "porównanie": f"{a}  vs  {b}",
        "mean_diff":  da.mean() - db.mean(),
        "median_diff": np.median(diff),
        "W": W, "Z": z, "p_raw": p,
        "p_bonferroni": min(p * n_comp, 1.0),
        "effect_r": r_eff,
        "istotne_0.05": (min(p * n_comp, 1.0) < 0.05),
    })
wil = pd.DataFrame(wil_rows)

wil.to_csv(OUTDIR_CMP / "wilcoxon_pairwise.csv", index=False)


_colors = {"GLM OLS": "darkorange", "GLM OLS+SS": "steelblue", "GLM AR-IRLS": "seagreen"}
cols = [_colors.get(m, None) for m in method_names]

fig, ax = plt.subplots(figsize=(max(8, N*1.1), 5))
x = np.arange(N); w = 0.8 / k
for j, m in enumerate(method_names):
    ax.bar(x + (j - (k-1)/2)*w, acc_disp[m].values, w, label=m,
           color=_colors.get(m), alpha=.88)
ax.axhline(0.5, color="red", ls="--", lw=1, label="poziom losowy")
ax.set_xticks(x); ax.set_xticklabels(acc_disp.index, rotation=35, ha="right")
ax.set_ylabel("LOSO accuracy"); ax.set_ylim(0, 1)
ax.set_title(f"Accuracy per pacjent — sLDA, {HRF_NAME}, drift={DRIFT}")
ax.legend(ncol=2, fontsize=8); plt.tight_layout()
fig.savefig(OUTDIR_CMP / "fig_accuracy_per_pacjent.pdf"); plt.show(); plt.close(fig)


fig, ax = plt.subplots(figsize=(7, 5))
data = [acc_wide[m].values for m in method_names]
bp = ax.boxplot(data, labels=method_names, showmeans=True, patch_artist=True)
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c); patch.set_alpha(.35)
for j, m in enumerate(method_names):
    ax.scatter(np.full(N, j+1) + np.random.uniform(-.06, .06, N),
               acc_wide[m].values, color=_colors.get(m), s=22, zorder=3, alpha=.8)
ax.axhline(0.5, color="red", ls="--", lw=1)
ax.set_ylabel("LOSO accuracy"); ax.set_ylim(0, 1)
ax.set_title(f"Rozkład accuracy (3 metody) — sLDA, {HRF_NAME}, d{DRIFT}")
plt.xticks(rotation=15); plt.tight_layout()
fig.savefig(OUTDIR_CMP / "fig_boxplot_accuracy.pdf"); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
for s in acc_disp.index:
    ax.plot(range(k), acc_disp.loc[s, method_names].values,
            marker="o", lw=1, alpha=.5)
ax.plot(range(k), [acc_wide[m].mean() for m in method_names],
        "k-o", lw=3, label="średnia")
ax.set_xticks(range(k)); ax.set_xticklabels(method_names, rotation=15)
ax.set_ylabel("LOSO accuracy"); ax.set_ylim(0, 1)
ax.set_title("Parowane wyniki pacjentów między metodami")
ax.legend(); plt.tight_layout()
fig.savefig(OUTDIR_CMP / "fig_paired_lines.pdf"); plt.show(); plt.close(fig)

